# Reto 1: El Validador de Respuestas RAG (Nivel: Básico)

Este notebook te guiará a través del proceso de creación de un validador de respuestas RAG para un bot de atención al cliente de un banco, utilizando `deepeval` y `pytest` con el modelo Gemini `gemini-3.1-flash-lite`.

## Configuración de la API de Gemini

Para usar la API de Gemini, necesitarás una clave API. Si aún no tienes una, crea una clave en [Google AI Studio](https://makersuite.google.com/app/apikey).

En Colab, añade la clave al gestor de secretos (el icono de la llave 🔑 en el panel izquierdo). Nómbrala `GOOGLE_API_KEY`. Luego, pasa la clave al SDK:

In [ ]:
import os
import google.generativeai as genai
from google.colab import userdata

# Obtener la clave API de Colab secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Configurar la API de Gemini
genai.configure(api_key=GOOGLE_API_KEY)

# Inicializar el modelo Gemini 3.1 Flash Lite
gemini_model = genai.GenerativeModel('gemini-3.1-flash-lite')
print("Gemini API configurada y modelo 'gemini-3.1-flash-lite' inicializado.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Gemini API configurada y modelo 'gemini-3.1-flash-lite' inicializado.


## Instalación de Dependencias

Instalaremos `deepeval` y `pytest` para la evaluación del modelo.

In [ ]:
!pip install deepeval pytest pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 965.1/965.1 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.4/267.4 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 2.6 MB/s eta 0:00:00


## Simulación del Bot de Atención al Cliente

Aquí definimos una función que simula el bot. Este bot intentará responder a las preguntas usando un contexto proporcionado, pero tiene una tendencia a 'alucinar' tasas de interés si no se le proporciona el contexto exacto.

In [ ]:
def bank_customer_service_bot(query, context=None):
    """
    Simula un bot de atención al cliente de un banco.
    A veces 'alucina' con las tasas de interés si no hay contexto específico.
    """
    prompt = f"""Eres un asistente de atención al cliente de un banco. Responde a la siguiente pregunta del usuario de la manera más útil y concisa posible.
    Si la pregunta es sobre tasas de interés y no hay contexto específico, puedes inventar una.

    Contexto:
    {context if context else 'No hay contexto disponible.'}

    Pregunta del usuario: {query}

    Respuesta:"""

    try:
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Lo siento, hubo un error al procesar tu solicitud: {e}"


## Preparación de los Datos para la Evaluación

Necesitamos preguntas, respuestas esperadas y contextos para evaluar el bot. Los contextos son cruciales para las métricas de `Faithfulness` y `Answer Relevancy`.

In [ ]:
import pandas as pd

# Datos para la evaluación (5 casos de prueba)
test_cases_data = [
    {
        "query": "¿Cuáles son los requisitos para abrir una cuenta de ahorros?",
        "expected_answer": "Para abrir una cuenta de ahorros, necesitas una identificación oficial vigente, comprobante de domicilio no mayor a 3 meses y un depósito inicial mínimo de $100.",
        "context": "Para abrir una cuenta de ahorros, se requiere identificación oficial, comprobante de domicilio y un depósito inicial de $100."
    },
    {
        "query": "¿Cuál es la tasa de interés para un préstamo hipotecario?",
        "expected_answer": "La tasa de interés para un préstamo hipotecario es del 7.5% anual, sujeta a evaluación crediticia.",
        "context": "Nuestras tasas de interés para préstamos hipotecarios comienzan en 7.5% anual."
    },
    {
        "query": "¿Cómo puedo solicitar una tarjeta de crédito?",
        "expected_answer": "Puedes solicitar una tarjeta de crédito en línea a través de nuestra página web o visitando cualquiera de nuestras sucursales con tu identificación y comprobante de ingresos.",
        "context": "La solicitud de tarjetas de crédito se puede realizar en línea o en sucursal con identificación y comprobante de ingresos."
    },
    {
        "query": "¿Cuáles son las comisiones por transferencia internacional?",
        "expected_answer": "Las transferencias internacionales tienen una comisión del 1% del monto transferido, con un mínimo de $25 y un máximo de $150.",
        "context": "Las comisiones por transferencias internacionales son del 1% del monto, con un mínimo de $25 y un máximo de $150."
    },
    {
        "query": "¿Cuál es la tasa de interés de los depósitos a plazo fijo?",
        "expected_answer": "La tasa de interés de los depósitos a plazo fijo puede variar, pero actualmente es del 3% anual para plazos de 1 año.",
        "context": "No se proporcionó información específica sobre tasas de interés de depósitos a plazo fijo. El bot podría alucinar aquí."
    }
]

# Un caso de prueba para provocar una alucinación sobre tasas de interés (sin contexto)
hallucination_test_case = {
    "query": "¿Qué tasa de interés ofrecen para depósitos a 3 años?",
    "expected_answer": "Actualmente, ofrecemos una tasa de interés del 4% anual para depósitos a 3 años.", # Esto es inventado para el propósito de la prueba
    "context": None # Sin contexto explícito
}

test_cases_data.append(hallucination_test_case)

df_test_cases = pd.DataFrame(test_cases_data)
display(df_test_cases)

,query,expected_answer,context
0,¿Cuáles son los requisitos para abrir una cuen...,"Para abrir una cuenta de ahorros, necesitas un...","Para abrir una cuenta de ahorros, se requiere ..."
1,¿Cuál es la tasa de interés para un préstamo h...,La tasa de interés para un préstamo hipotecari...,Nuestras tasas de interés para préstamos hipot...
2,¿Cómo puedo solicitar una tarjeta de crédito?,Puedes solicitar una tarjeta de crédito en lín...,La solicitud de tarjetas de crédito se puede r...
3,¿Cuáles son las comisiones por transferencia i...,Las transferencias internacionales tienen una ...,Las comisiones por transferencias internaciona...
4,¿Cuál es la tasa de interés de los depósitos a...,La tasa de interés de los depósitos a plazo fi...,No se proporcionó información específica sobre...
5,¿Qué tasa de interés ofrecen para depósitos a ...,"Actualmente, ofrecemos una tasa de interés del...",None


## Creación del Archivo de Pruebas (`test_bot.py`)

Ahora crearemos un archivo Python (`test_bot.py`) que contendrá nuestras pruebas usando `pytest` y `deepeval`. Este archivo se guardará en el entorno de Colab.

In [ ]:
%%writefile test_bot.py

import pytest
import os
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval import assert_test
from deepeval.models import DeepEvalBaseLLM

# --- BEGIN: Esto simula la importación del bot del entorno de Colab ---
# En un entorno real, descomentarías esto y lo importarías de un archivo.
# from your_bot_module import bank_customer_service_bot
# from your_gemini_module import gemini_model

# Re-definir bot y modelo para que pytest pueda ejecutarlos en un script separado
import google.generativeai as genai

# Obtener la clave API de las variables de entorno
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in environment variables. Please set it.")

genai.configure(api_key=GOOGLE_API_KEY)
gemini_model = genai.GenerativeModel('gemini-3.1-flash-lite')

# Custom wrapper for Google Gemini model to be compatible with deepeval
class CustomGeminiModel(DeepEvalBaseLLM):
    def __init__(self, model):
        self.model = model
        self.model_name = model.model_name # Assign model_name attribute

    def load_model(self):
        # Model is already loaded, so return it directly
        return self.model

    # Implement the abstract methods from DeepEvalBaseLLM
    def generate(self, prompt: str) -> str:
        loaded_model = self.load_model()
        response = loaded_model.generate_content(prompt)
        return response.text

    async def a_generate(self, prompt: str) -> str:
        # Asynchronous generation if needed, otherwise defer to synchronous
        return self.generate(prompt)

    def get_model_name(self):
        return self.model_name

# Instantiate our custom deepeval-compatible Gemini model
deep_eval_gemini_wrapper = CustomGeminiModel(gemini_model)


def bank_customer_service_bot(query, context=None):
    prompt = f"""Eres un asistente de atención al cliente de un banco. Responde a la siguiente pregunta del usuario de la manera más útil y concisa posible.
    Si la pregunta es sobre tasas de interés y no hay contexto específico, puedes inventar una.

    Contexto:
    {context if context else 'No hay contexto disponible.'}

    Pregunta del usuario: {query}

    Respuesta:"""
    try:
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Lo siento, hubo un error al procesar tu solicitud: {e}"
# --- END: Simulación de importación ---


@pytest.fixture(scope="module")
def bot_fixture():
    return bank_customer_service_bot

@pytest.fixture(scope="module")
def test_cases():
    return [
        {
            "query": "¿Cuáles son los requisitos para abrir una cuenta de ahorros?",
            "expected_answer": "Para abrir una cuenta de ahorros, necesitas una identificación oficial vigente, comprobante de domicilio no mayor a 3 meses y un depósito inicial mínimo de $100.",
            "context": "Para abrir una cuenta de ahorros, se requiere identificación oficial, comprobante de domicilio y un depósito inicial de $100."
        },
        {
            "query": "¿Cuál es la tasa de interés para un préstamo hipotecario?",
            "expected_answer": "La tasa de interés para un préstamo hipotecario es del 7.5% anual, sujeta a evaluación crediticia.",
            "context": "Nuestras tasas de interés para préstamos hipotecarios comienzan en 7.5% anual."
        },
        {
            "query": "¿Cómo puedo solicitar una tarjeta de crédito?",
            "expected_answer": "Puedes solicitar una tarjeta de crédito en línea a través de nuestra página web o visitando cualquiera de nuestras sucursales con tu identificación y comprobante de ingresos.",
            "context": "La solicitud de tarjetas de crédito se puede realizar en línea o en sucursal con identificación y comprobante de ingresos."
        },
        {
            "query": "¿Cuáles son las comisiones por transferencia internacional?",
            "expected_answer": "Las transferencias internacionales tienen una comisión del 1% del monto transferido, con un mínimo de $25 y un máximo de $150.",
            "context": "Las comisiones por transferencias internacionales son del 1% del monto, con un mínimo de $25 y un máximo de $150."
        },
        {
            "query": "¿Cuál es la tasa de interés de los depósitos a plazo fijo?",
            "expected_answer": "La tasa de interés de los depósitos a plazo fijo puede variar, pero actualmente es del 3% anual para plazos de 1 año.",
            "context": None # Simular que no hay contexto para esta pregunta de tasa de interés para ver la alucinación
        }
    ]

def run_deepeval_test_case(bot_response, query, expected_answer, context, faithfulness_threshold, relevancy_threshold):
    # Pass our deepeval-compatible gemini wrapper to the metrics
    faithfulness_metric = FaithfulnessMetric(threshold=faithfulness_threshold, model=deep_eval_gemini_wrapper)
    answer_relevancy_metric = AnswerRelevancyMetric(threshold=relevancy_threshold, model=deep_eval_gemini_wrapper)

    test_case = LLMTestCase(
        input=query,
        actual_output=bot_response,
        expected_output=expected_answer,
        retrieval_context=[context] if context else []
    )

    assert_test(test_case, [faithfulness_metric, answer_relevancy_metric])


class TestBankBot:

    def test_requirements_savings_account(self, bot_fixture, test_cases):
        case = test_cases[0]
        bot_response = bot_fixture(case["query"], case["context"])
        run_deepeval_test_case(bot_response, case["query"], case["expected_answer"], case["context"],
                               faithfulness_threshold=0.7, relevancy_threshold=0.7)

    def test_mortgage_interest_rate(self, bot_fixture, test_cases):
        case = test_cases[1]
        bot_response = bot_fixture(case["query"], case["context"])
        run_deepeval_test_case(bot_response, case["query"], case["expected_answer"], case["context"],
                               faithfulness_threshold=0.7, relevancy_threshold=0.7)

    def test_credit_card_application(self, bot_fixture, test_cases):
        case = test_cases[2]
        bot_response = bot_fixture(case["query"], case["context"])
        run_deepeval_test_case(bot_response, case["query"], case["expected_answer"], case["context"],
                               faithfulness_threshold=0.7, relevancy_threshold=0.7)

    def test_international_transfer_fees(self, bot_fixture, test_cases):
        case = test_cases[3]
        bot_response = bot_fixture(case["query"], case["context"])
        run_deepeval_test_case(bot_response, case["query"], case["expected_answer"], case["context"],
                               faithfulness_threshold=0.7, relevancy_threshold=0.7)

    # Este test está diseñado para fallar si el bot alucina una tasa de interés
    def test_fixed_deposit_interest_rate_hallucination(self, bot_fixture, test_cases):
        case = test_cases[4]
        # Aquí ajustamos el threshold de Faithfulness más alto para detectar alucinaciones
        # Si el bot inventa una tasa y no está en el contexto, Faithfulness debería ser bajo.
        # La AnswerRelevancy se puede mantener más permisiva si la respuesta es relevante a la pregunta.
        bot_response = bot_fixture(case["query"], case["context"])
        run_deepeval_test_case(bot_response, case["query"], case["expected_answer"], case["context"],
                               faithfulness_threshold=0.9, relevancy_threshold=0.5) # Threshold alto para Faithfulness

Overwriting test_bot.py


## Ejecución de las Pruebas

Ahora puedes ejecutar las pruebas usando `pytest` desde la línea de comandos en Colab. El comando `!pytest` buscará automáticamente el archivo `test_bot.py` y ejecutará los tests definidos.

In [ ]:
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
!pytest test_bot.py

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: xdist-3.8.0, repeat-0.9.4, deepeval-4.0.5, asyncio-1.4.0, rerunfailures-16.3, typeguard-4.5.2, langsmith-0.8.5, anyio-4.13.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 5 items                                                              

test_bot.py ..FFF                                                        [100%]Running teardown with pytest sessionfinish...


=================================== FAILURES ===================================
___________________ TestBankBot.test_credit_card_application ___________________

args = (model: "models/gemini-3.1-flash-lite"
contents {
  parts {
    text: "Given the text, breakdown and generate a list o...}\n  quota_value: 15\n}\n, retry_delay {\n  seconds: 38\n}\n]\n\nJSON:\n"
  }
  role:

## Interpretación de los Resultados

*   **Verde (PASSED)**: Indica que la respuesta del bot cumplió con los umbrales de `Faithfulness` y `Answer Relevancy`.
*   **Rojo (FAILED)**: Indica que la respuesta del bot no cumplió con los umbrales. En nuestro caso, el test `test_fixed_deposit_interest_rate_hallucination` debería fallar si el bot inventa una tasa de interés sin un contexto que la respalde, debido al umbral alto de `Faithfulness`.